Section B — Practical Coding Tasks

Task 1: Save & Load a Delivery Time Predictor


In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# -----------------------------
# Step 1: Generate Dataset
# -----------------------------

np.random.seed(42)

n = 100

distance_km = np.random.uniform(1,15,n)

num_items = np.random.randint(1,10,n)

rain_flag = np.random.randint(0,2,n)

delivery_time_min = (
    10
    + distance_km*3
    + num_items*2
    + rain_flag*5
    + np.random.normal(0,2,n)
)

df = pd.DataFrame({
    "distance_km":distance_km,
    "num_items":num_items,
    "rain_flag":rain_flag,
    "delivery_time_min":delivery_time_min
})

# -----------------------------
# Step 2: Split Data
# -----------------------------

X = df[["distance_km","num_items","rain_flag"]]

y = df["delivery_time_min"]

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# -----------------------------
# Step 3: Train Model
# -----------------------------

model = LinearRegression()

model.fit(X_train,y_train)

# -----------------------------
# Step 4: Evaluate Model
# -----------------------------

pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test,pred))

print("RMSE:",round(rmse,2))

# -----------------------------
# Step 5: Save Model
# -----------------------------

filename = "delivery_model.joblib"

joblib.dump(model,filename)

size_kb = os.path.getsize(filename)/1024

print(f"Model saved successfully as {filename}")

print(f"File Size: {size_kb:.2f} KB")

# -----------------------------
# Step 6: Load Model
# -----------------------------

loaded_model = joblib.load(filename)

print("Model loaded successfully!")

# -----------------------------
# Step 7: New Prediction
# -----------------------------

new_sample = pd.DataFrame({
    "distance_km":[8],
    "num_items":[4],
    "rain_flag":[1]
})

original_prediction = model.predict(new_sample)

loaded_prediction = loaded_model.predict(new_sample)

print("Original Prediction:",original_prediction)

print("Reloaded Prediction:",loaded_prediction)

# -----------------------------
# Step 8: Verify Predictions
# -----------------------------

if np.allclose(original_prediction,loaded_prediction):
    print("PASS")
else:
    print("FAIL")

RMSE: 2.44
Model saved successfully as delivery_model.joblib
File Size: 0.91 KB
Model loaded successfully!
Original Prediction: [47.13234424]
Reloaded Prediction: [47.13234424]
PASS


Task 2: Flask REST API for Order Predictions

In [2]:
# ================================
# Task 2: Flask REST API for Order Predictions
# Complete Code (Single Jupyter Cell)
# ================================

from flask import Flask, request, jsonify
import joblib
import pandas as pd

# --------------------------------
# Create Flask App
# --------------------------------
app = Flask(__name__)

# --------------------------------
# Load Model Once at Startup
# --------------------------------
try:
    model = joblib.load("delivery_model.joblib")
    print("✅ Model Loaded Successfully!")
except Exception as e:
    print("❌ Error Loading Model:", e)

# --------------------------------
# Home Route
# --------------------------------
@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Delivery Prediction API is Running!"
    })

# --------------------------------
# Prediction Route
# --------------------------------
@app.route("/predict", methods=["POST"])
def predict():

    # Get JSON Data
    data = request.get_json()

    # Check if JSON exists
    if data is None:
        return jsonify({
            "error": "No JSON data received."
        }), 400

    # Required Fields
    required_fields = [
        "distance_km",
        "num_items",
        "rain_flag"
    ]

    # Check Missing Fields
    missing_fields = []

    for field in required_fields:
        if field not in data:
            missing_fields.append(field)

    if len(missing_fields) > 0:
        return jsonify({
            "error": "Missing required fields.",
            "missing_fields": missing_fields
        }), 400

    try:
        # Create DataFrame
        sample = pd.DataFrame({
            "distance_km": [data["distance_km"]],
            "num_items": [data["num_items"]],
            "rain_flag": [data["rain_flag"]]
        })

        # Predict
        prediction = model.predict(sample)[0]

        # Return Result
        return jsonify({
            "predicted_delivery_time_min": round(float(prediction), 1)
        }), 200

    except Exception as e:
        return jsonify({
            "error": str(e)
        }), 400

# --------------------------------
# Run Flask App
# --------------------------------
app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

✅ Model Loaded Successfully!
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.2.105:5000
Press CTRL+C to quit
127.0.0.1 - - [19/Jul/2026 14:01:29] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Jul/2026 14:01:29] "GET /favicon.ico HTTP/1.1" 404 -


Task 3: LLM-Powered Dish Description Generator

In [ ]:
# ==========================================================
# Task 3 : LLM-Powered Dish Description Generator
# Jupyter Notebook (.ipynb) - Single Cell Version
# ==========================================================

import requests
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

# -----------------------------
# Ollama Configuration
# -----------------------------
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3.2"

# -----------------------------
# UI Widgets
# -----------------------------
dish = widgets.Text(
    value="Paneer Tikka Masala",
    description="Dish:"
)

cuisine = widgets.Text(
    value="North Indian",
    description="Cuisine:"
)

length = widgets.Dropdown(
    options=["Short","Medium","Long"],
    value="Medium",
    description="Length:"
)

generate_btn = widgets.Button(
    description="Generate",
    button_style="success"
)

regen_btn = widgets.Button(
    description="Regenerate",
    button_style="warning"
)

output = widgets.Output()

# -----------------------------
# Prompt Builder
# -----------------------------
def build_prompt():
    return f"""
You are an expert food marketing copywriter.

Dish Name: {dish.value}
Cuisine Type: {cuisine.value}
Description Length: {length.value}

Write an appetising, customer-facing promotional description.
Highlight taste, aroma, freshness, and premium ingredients.
Do not use bullet points.
"""

# -----------------------------
# LLM Call
# -----------------------------
def generate_description(_):

    with output:
        clear_output()

        prompt = build_prompt()

        # Print prompt in terminal (assignment requirement)
        print("\n========== FINAL PROMPT ==========")
        print(prompt)
        print("==================================\n")

        print("Generating description...\n")

        payload = {
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False
        }

        try:
            response = requests.post(
                OLLAMA_URL,
                json=payload,
                timeout=120
            )

            response.raise_for_status()

            result = response.json()["response"]

            print("Generated Description\n")
            display(Markdown(result))

            print("\nCharacter Count:", len(result))

        except Exception as e:
            print("Error:", e)

# -----------------------------
# Button Events
# -----------------------------
generate_btn.on_click(generate_description)
regen_btn.on_click(generate_description)

# -----------------------------
# Display UI
# -----------------------------
display(
    widgets.VBox([
        dish,
        cuisine,
        length,
        widgets.HBox([generate_btn, regen_btn]),
        output
    ])
)

Task 4: LangChain Food Order Assistant with Memory

In [9]:
!pip install langchain langchain-community langchain-core
!pip install ollama

In [15]:
!ollama pull llama3.2


'ollama' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
# =====================================================
# Task 4: LangChain Food Order Assistant with Memory
# Complete Single Cell (Jupyter Notebook)
# =====================================================

from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import ConversationChain
from langchain_community.llms import Ollama

# -----------------------------------------------------
# Load LLM
# -----------------------------------------------------

llm = Ollama(model="llama3.2")

# -----------------------------------------------------
# Conversation Memory
# -----------------------------------------------------

memory = ConversationBufferMemory(
    memory_key="history",
    return_messages=False
)

# -----------------------------------------------------
# Prompt Template
# -----------------------------------------------------

prompt = PromptTemplate(
    input_variables=["history", "input"],
    template="""
You are QuickBite Food Delivery Assistant.

Rules:
- Be polite and helpful.
- Remember the customer's delivery address once they mention it.
- Remember the customer's dietary preference.
- Use this information naturally in later replies.
- Continue remembering even if the topic changes.

Conversation History:
{history}

User:
{input}

QuickBite:
"""
)

# -----------------------------------------------------
# Conversation Chain
# -----------------------------------------------------

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    prompt=prompt,
    verbose=False
)

print("Memory Key:", memory.memory_key)
print("-" * 60)

# -----------------------------------------------------
# Simulated Conversation (5 Turns)
# -----------------------------------------------------

conversation_list = [

    "My delivery address is 25 MG Road, Ahmedabad.",

    "I am vegetarian.",

    "Can you recommend something for dinner?",

    "Can you send it to my address and remember I don't eat meat?",

    "By the way, what's today's weather?"

]

turns = 0

for msg in conversation_list:

    turns += 1

    print(f"\nUser: {msg}")

    reply = conversation.predict(input=msg)

    print(f"QuickBite: {reply}")

print("\nType 'exit' to finish.")

while True:

    user = input("You: ")

    if user.lower() == "exit":

        print("\nConversation Ended")

        print(f"\nTotal Turns : {turns}")

        print("\n========== MEMORY BUFFER ==========\n")

        print(memory.buffer)

        break

    turns += 1

    answer = conversation.predict(input=user)

    print("\nQuickBite:", answer)